# Using the firmware reference

Having documentation still leaves a challenge: how do the commands fit together, and does the device behave as described?

Use the **Plate Reader — Command Reference** PDF, evidence from outside the manual, and experiments on the Pico to:

- **Find the two deliberate mistakes in the manual**, with evidence for each correction.
- **Build `read_absorbance(wavelength=600)`** by combining the supplied functions.

You can start here even if you did not finish the earlier notebooks. All individual command functions and both decoders are supplied below. Your task is to work out which calls belong together and in what order.

Keep the manual open beside this notebook. Run the setup cells with **Shift+Enter**, then use the experiment cells to investigate.

The Pico's onboard LED is on when the simulated door is open and off when it is closed.


## 1. Read the manual alongside other evidence

Follow a command through the manual: find its identifier and parameter, read its reply format, then look for requirements elsewhere in the text. The examples below show how those details become Python code.

A byte-format example tells you how to send a command. The surrounding prose explains when to send it and what it changes. Both need checking against the device.

Use evidence outside the manual to check its claims:

| Source | What it can help establish |
| --- | --- |
| [Door capture](captures/door.txt) and [door practical](02_door_practical.ipynb) | Which commands move the reader and how state replies describe its position |
| [Reading capture](captures/reading.txt) and [reading practical](03_reading_practical.ipynb) | Commands used for a successful luminescence measurement |
| [Absorbance capture](captures/absorbance.txt) and [vendor software practical](04_vendor_software_practical.ipynb) | A working absorbance sequence and how its reply becomes values |
| The Pico | Whether a proposed sequence works, and what changes after a command |

The supplied functions carry forward the earlier practicals. Compare their command bytes with the manual too. A working capture shows one successful sequence; experiments can establish which steps are required.

## 2. Connect your Pico

The manual's **Connecting and sending commands** section says:

> Connect to the USB serial port at **115200 baud, 8N1, no flow control**.
> Clear any startup text before sending the first command.

This tells us how to open the connection: `baudrate=115200` sets the speed. `8N1` means eight data bits, no parity, and one stop bit; these are pyserial's defaults. Flow control is off by default too. `reset_input_buffer()` clears startup text.

Close the serial connection in any previous notebook with `pico.close()`. Disconnect the desktop application or any other program using the Pico.

Replace `YOUR_PORT` below with your port, such as `COM3`, `/dev/cu.usbmodem...`, or `/dev/ttyACM0`. Run the cell on the computer connected to the Pico.

If `import serial` fails, uncomment and run the installation line first.

In [ ]:
# %pip install pyserial
import serial
import struct
import time

PORT = "YOUR_PORT"
pico = serial.Serial(PORT, baudrate=115200, timeout=1, write_timeout=1)
time.sleep(1)
pico.reset_input_buffer()
print("Connected to", pico.port)

### Read a command format

In **Connecting and sending commands**, the manual gives this format:

```text
A7 03 [command] 00 [parameter: 2 bytes] 00 FF
```

Read the fixed bytes literally. The bracketed fields are places to insert values. The format describes **eight bytes** altogether.

The same section says:

> Use `00 00` when a command has no parameter.

For a two-byte parameter it says to send the **high byte first**. Use `struct.pack(">H", argument)` to create those two bytes. The function below supplies this conversion for you.

Now look at the **Wavelength** example under **Measurement settings**:

```text
Send:     A7 03 01 00 01 C2 00 FF
```

Match it to the format: `01` is the command, and `01 C2` is the parameter, 450 nm. The Python call is `send_command(0x01, 450)`. The example supplies one wavelength; the allowed range comes from the surrounding text.

Run the function below as supplied. `reply_length` is how many bytes to receive. It defaults to 8; the measurement and state sections specify longer replies.

The manual allows one second for a response. With our one-second timeout, `read()` waits for the requested number of bytes or returns what arrived when time runs out. An eight-byte error stays eight bytes even if we requested 390.

In [ ]:
def send_command(command, argument=0, reply_length=8):
    prefix = bytes.fromhex("A7 03") + struct.pack("B", command) + bytes.fromhex("00")
    argument_bytes = struct.pack(">H", argument)
    ending = bytes.fromhex("00 FF")
    request = prefix + argument_bytes + ending

    pico.write(request)
    reply = pico.read(reply_length)
    return reply

### Read the reply format before decoding data

**Connecting and sending commands** says:

> Responses begin with `A7 83`, followed by the command code and a status byte.
> Status `00` means the command succeeded.

Count from zero in Python: the two header bytes occupy indices 0 and 1, the command is at 2, and the status is at **3**.

The **Error responses** section adds:

> Check the status first: a failed read returns an 8-byte error, not plate data.

That is why the supplied decoders call `require_success()` first. An `assert` stops the cell and shows its message if a check fails. During an experiment, print the raw reply and `reply[3]` before trying to decode it.

In [ ]:
def require_success(reply):
    assert len(reply) >= 8, "Incomplete reply: check the connection."
    assert reply[-2:] == bytes.fromhex("00 FF"), "Unexpected reply ending."
    assert reply[3] == 0, "The reader rejected the command. Look up reply[3] in the manual."
    return reply

### Find a command's meaning and requirements

A command entry contains more than its byte code. Under **Opening and closing**, for example:

> Measurements are unavailable while it is open.

This describes a requirement for reading. The same section also says:

> Opening and closing preserve the measurement settings.

An acknowledgement tells you the command succeeded. The prose tells you what should change and what should stay the same. Look for words such as **before**, **preserves**, **clears**, and **defaults** in the other entries too.

These supplied functions each send one command and return its reply. Compare the manual's claims with the functions and later with the device.

| Function | Manual entry |
| --- | --- |
| `reset()` | **Reset** |
| `open_reader()` | **Opening and closing** |
| `close_reader()` | **Opening and closing** |
| `set_wavelength(450)` | Wavelength under **Measurement settings** |
| `set_gain(2)` | Gain under **Measurement settings** |
| `request_state()` | **Reader state — GET_STATE** |

Most replies are eight bytes. `request_state()` asks for 12 bytes. Run the cell to define the functions; defining them sends no commands.

In [ ]:
def reset():
    reply = send_command(0x04)
    return require_success(reply)


def open_reader():
    reply = send_command(0x06)
    return require_success(reply)


def close_reader():
    reply = send_command(0x07)
    return require_success(reply)


def set_wavelength(wavelength):
    if type(wavelength) is not int or not 340 <= wavelength <= 850:
        raise ValueError("Use a whole-number wavelength from 340 to 850 nm.")
    reply = send_command(0x01, wavelength)
    return require_success(reply)


def set_gain(gain):
    if type(gain) is not int or not 1 <= gain <= 16:
        raise ValueError("Use a whole-number gain from 1 to 16.")
    reply = send_command(0x31, gain)
    return require_success(reply)


def request_state():
    reply = send_command(0x02, reply_length=12)
    require_success(reply)
    assert len(reply) == 12, "Expected 12 bytes for reader state."
    return reply

### Find the measurement command and reply length

Under **Measurements**, the **Absorbance — READ_ABSORBANCE (`21`)** entry shows:

```text
Send:     A7 03 21 00 00 00 00 FF
Receive:  A7 83 21 00 [96 A values] [96 B values] 00 FF
```

The entry also says:

> The response is **390 bytes**. It contains all A values first, then all B values.

The command ID gives us `0x21`; the stated reply length gives us `reply_length=390`. The **Luminescence — READ_LUMINESCENCE (`20`)** entry gives `0x20` and 198 bytes in the same way.

Each function below sends just the measurement command and returns its reply bytes. The `_command` suffix means it sends one command and returns reply bytes; it does not prepare the reader or decode the result. You choose any preparation calls separately. The `Send`/`Receive` pair describes one exchange; look elsewhere in the manual for the conditions needed for it to succeed.

Either measurement can return an eight-byte error. Its format is in **Error responses**.

In [ ]:
def read_luminescence_command():
    return send_command(0x20, reply_length=198)


def read_absorbance_command():
    return send_command(0x21, reply_length=390)

### Turn the reply layout into a decoder

The **Measurements** section says:

> Each returned number occupies two bytes, high byte first.

For absorbance, it then says:

> For each well, divide its A value by its B value:

Together with the reply format above, this explains the supplied decoder:

| Manual detail | Python |
| --- | --- |
| Four bytes before the data: `A7 83 21 00` | Start at index `4` |
| 96 A values, two bytes each: 192 bytes | `reply[4:196]` |
| Then 96 B values: another 192 bytes | `reply[196:388]` |
| Two-byte integers, high byte first | `struct.unpack(">96H", ...)` |
| Divide each A by its corresponding B | `first[index] / second[index]` |

Python slices exclude the end index. The lengths add up: 4 header bytes + 192 A bytes + 192 B bytes + 2 ending bytes = **390**.

Luminescence has just one block of 96 numbers: 4 + 192 + 2 = **198** bytes. Its values can be used directly.

Run the supplied cell to define both decoders.

In [ ]:
def decode_luminescence(reply):
    require_success(reply)
    assert len(reply) == 198, "Expected 198 bytes for a luminescence reading."
    assert reply[:4] == bytes.fromhex("A7 83 20 00"), "Unexpected luminescence reply."
    values = struct.unpack(">96H", reply[4:196])
    return list(values)


def decode_absorbance(reply):
    require_success(reply)
    assert len(reply) == 390, "Expected 390 bytes for an absorbance reading."
    assert reply[:4] == bytes.fromhex("A7 83 21 00"), "Unexpected absorbance reply."

    first = struct.unpack(">96H", reply[4:196])
    second = struct.unpack(">96H", reply[196:388])
    values = []
    for index in range(96):
        assert second[index] != 0, "Cannot divide by zero."
        values.append(first[index] / second[index])
    return values

## 3. Investigate the sequence and find both mistakes

Read **Reset** before choosing a starting point:

> RESET clears wavelength, and restores gain 1.
> It leaves the reader's open/closed position unchanged.

This tells you which parts of the state a reset makes predictable and which part can carry over from previous work. Disconnecting is different: the same section says reconnecting USB does not reset the reader.

Start with a measurement after reset, then use its reply to decide what to investigate next.

In [ ]:
reset()
reply = read_absorbance_command()
print("Reply:", reply.hex(" "))
print("Length:", len(reply))
print("Status:", reply[3])

### Read a state reply using its field layout

**Reader state — GET_STATE** gives this layout:

```text
A7 83 02 00 [wavelength: 2 bytes] [gain: 2 bytes] [wavelength set] [open] 00 FF
```

Start after the four header bytes and count each field's width:

| Field | Where it is in `state = request_state()` |
| --- | --- |
| Wavelength | `struct.unpack(">H", state[4:6])[0]` |
| Gain | `struct.unpack(">H", state[6:8])[0]` |
| Wavelength set | `state[8]` |
| Open | `state[9]` |

The manual's field table defines the open field as:

> `00` = closed; `01` = open

For example, `print(state[9])` displays that field. Request a fresh state after each command you want to investigate. This lets you compare the resulting state with the action promised in the manual.

### Use the error table to choose an experiment

The **Error responses** section says:

> An error response is always **8 bytes**. The fourth byte tells you what went wrong:

Read your printed status, then find its row in that table. Python prints decimal numbers; for example, Python's `5` corresponds to `05` in this manual. The table's **What to do** column suggests a next action. Treat that suggestion as a claim you can check.

Add your next call in the cell below, request a measurement again, and inspect the reply. Repeat until you have a successful measurement. A call has the form `set_wavelength(450)`; each call goes on its own line.

As you investigate:

- Compare the manual's claims with the captures and earlier practicals.
- Change one thing at a time so you can tell what affected the result.
- Use `request_state()` to check what a command actually changed (see the field guide below).
- Try leaving out a step that the manual says is required. Use `reset()` before comparing sequences so old settings do not affect the conclusion.

A zero status means the command was accepted. Check the resulting state or measurement to establish what it did.

In [ ]:
# Add the command calls you want to try here.

reply = read_absorbance_command()
print("Reply:", reply.hex(" "))
print("Length:", len(reply))
print("Status:", reply[3])

When the measurement status is zero, run the cell below to decode it. A successful absorbance reply contains 96 results.

In [ ]:
readings = decode_absorbance(reply)
print("Number of readings:", len(readings))
print("First three:", [round(value, 4) for value in readings[:3]])

### Two corrections, supported by evidence

Finding both mistakes is part of the main challenge. For each one, identify:

- The exact claim in the manual, its section, and the corrected behaviour.
- The capture, earlier result, or supplied code that led you to question it.
- A device experiment that supports the correction.

Compare the examples with the prose as well as with the device. Check **Opening and closing**, **Measurement settings**, and the related examples in **Reader state — GET_STATE** and advice in **Error responses**.

A successful reading alone does not establish that every preparation step was necessary. Likewise, an acknowledgement alone does not establish that a command performed the action named in the manual.

In [ ]:
# Use this cell for further experiments to check the two claims.

## 4. Build the reading function

Turn your successful sequence into `read_absorbance(wavelength=600)`.

The function should:

- Work after reset and when the reader starts open.
- Use the requested wavelength and keep the current gain.
- Return a list of 96 absorbance values.

Use the supplied command functions and `decode_absorbance()`. Add each call on its own indented line. Use `wavelength` for the caller's chosen value. The last line should `return` the decoded values.

The manual describes individual commands. Choosing the sequence is your part: use the evidence you gathered, including your two corrections.

In [ ]:
def read_absorbance(wavelength=600):
    # Add your preparation, measurement, and decoding calls here.
    pass

### Try your function

After completing the function, run these cells. The first starts with a reset and an open reader. The next compares two wavelengths and repeats a measurement.

In [ ]:
reset()
open_reader()
readings = read_absorbance(wavelength=600)
print("Number of readings:", len(readings))
print("First three:", [round(value, 4) for value in readings[:3]])

In [ ]:
readings_450 = read_absorbance(wavelength=450)
print("450 nm:", [round(value, 4) for value in readings_450[:3]])
print("Same as 600 nm:", readings_450 == readings)
print("Same when repeated:", read_absorbance(wavelength=450) == readings_450)

## 5. Compare the evidence

Which parts of your sequence came from the manual? Which needed another source or an experiment? How did each of the two mistakes affect your approach?

You now have a reading function whose sequence is supported by evidence, and two specific corrections to the manual. Documentation helped explain the commands; combining them and checking their behaviour still required investigation.

## 6. Disconnect

Run this when you finish. It releases the serial port for other notebooks or the desktop application.

In [ ]:
pico.close()
print("Serial connection closed.")